In [1]:
import json
import faiss
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from transformers import pipeline

C:\Users\siddh\TruthLens\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

nli_model = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli"
)

print("Models Loaded")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3845.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 515/515 [00:00<00:00, 2104.25it/s]


Models Loaded


In [11]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross Encoder Loaded")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 3078.92it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross Encoder Loaded


In [3]:
with open(
    "corpus.json",
    "r",
    encoding="utf-8"
) as f:
    corpus = json.load(f)

index = faiss.read_index(
    "faiss_index.index"
)

print("Corpus:", len(corpus))
print("FAISS :", index.ntotal)

Corpus: 20902
FAISS : 20897


In [4]:
claims = [
    "A fusion reactor in South Korea generated sustained energy for 48 hours in 2026",
    "India launched a nationwide AI surveillance system in 2026",
    "A solar flare disrupted global satellite communication networks in 2026"
]

In [5]:
def retrieve_evidence(
    claim,
    top_k=5
):

    claim_embedding = embedding_model.encode(
        [claim]
    )

    distances, indices = index.search(
        np.array(claim_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(
            corpus[idx]
        )

    return results

In [12]:
def rerank_evidence(claim, evidences):

    pairs = [
        (claim, evidence)
        for evidence in evidences
    ]

    scores = cross_encoder.predict(pairs)

    ranked = sorted(
        zip(evidences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked

In [7]:
def numeric_mismatch(
    claim,
    evidence
):

    claim_nums = re.findall(
        r"\d+",
        claim
    )

    evidence_nums = re.findall(
        r"\d+",
        evidence
    )

    if claim_nums and evidence_nums:

        if claim_nums != evidence_nums:
            return True

    return False

In [6]:
def verify_pair(
    claim,
    evidence
):

    if numeric_mismatch(
        claim,
        evidence
    ):
        return {
            "label":"CONTRADICTION",
            "score":0.90
        }

    result = nli_model(
        f"{claim} </s></s> {evidence}"
    )[0]

    return {
        "label":result["label"],
        "score":round(
            result["score"],
            3
        )
    }

In [13]:
def verify_claim(claim):

    evidences = retrieve_evidence(claim)

    ranked = rerank_evidence(
        claim,
        evidences
    )

    # keep best 3 only
    top_evidences = [
        evidence
        for evidence, score in ranked[:3]
    ]

    results = []

    for evidence in top_evidences:

        result = verify_pair(
            claim,
            evidence
        )

        results.append({
            "evidence": evidence,
            "label": result["label"],
            "score": result["score"]
        })

    return results

In [14]:
def final_decision(
    results
):

    entail = 0
    contra = 0
    neutral = 0

    for r in results:

        label = r["label"]

        score = r["score"]

        if label == "ENTAILMENT":
            entail += score

        elif label == "CONTRADICTION":
            contra += score

        else:
            neutral += score

    if contra > entail and contra > neutral:
        return "CONTRADICTED"

    if entail > contra and entail > neutral:
        return "SUPPORTED"

    return "NEUTRAL"

In [18]:
all_decisions = []

for claim in claims:

    results = verify_claim(claim)

    decision = final_decision(results)

    all_decisions.append(decision)

    explanation = explain_decision(
        claim,
        results,
        decision
    )

    print("\n====================")
    print("CLAIM:")
    print(explanation["claim"])

    print("\nFINAL:")
    print(explanation["decision"])

    print("\nCONFIDENCE:")
    print(explanation["confidence"])

    print("\nBEST EVIDENCE:")
    print(explanation["evidence"])

    print("\nALL EVIDENCE:")

    for r in results:

        print(
            f"{r['label']} "
            f"({r['score']})"
        )

        print("->", r["evidence"])


CLAIM:
A fusion reactor in South Korea generated sustained energy for 48 hours in 2026

FINAL:
CONTRADICTED

CONFIDENCE:
0.9

BEST EVIDENCE:
South Korea's KSTAR reactor has achieved a new milestone in nuclear fusion by maintaining superheated plasma at 100 million°C for 48 seconds, ...

ALL EVIDENCE:
CONTRADICTION (0.9)
-> South Korea's KSTAR reactor has achieved a new milestone in nuclear fusion by maintaining superheated plasma at 100 million°C for 48 seconds, ...
CONTRADICTION (0.9)
-> In 2026, South Korea will achieve a historic energy milestone in plasma fusion, and the research team will break records that they had ...
CONTRADICTION (0.9)
-> South Korean researchers have maintained plasma temperatures of 100 million degrees Celsius for 48 seconds inside a tokamak fusion reactor.

CLAIM:
India launched a nationwide AI surveillance system in 2026

FINAL:
CONTRADICTED

CONFIDENCE:
0.978

BEST EVIDENCE:
The India AI Impact Summit 2026 is a flagship global gathering hosted by the Gov

In [16]:
def calculate_trust_score(decisions):

    total = len(decisions)

    supported = decisions.count("SUPPORTED")
    contradicted = decisions.count("CONTRADICTED")
    neutral = decisions.count("NEUTRAL")

    trust_score = (
        supported +
        (0.5 * neutral)
    ) / total * 100

    return {
        "supported": supported,
        "contradicted": contradicted,
        "neutral": neutral,
        "trust_score": round(trust_score, 2)
    }

In [17]:
def explain_decision(claim, results, final_label):

    best_evidence = max(
        results,
        key=lambda x: x["score"]
    )

    explanation = {
        "claim": claim,
        "decision": final_label,
        "confidence": best_evidence["score"],
        "evidence": best_evidence["evidence"]
    }

    return explanation

In [19]:
stats = calculate_trust_score(
    all_decisions
)

print("\n\n====================")
print("TRUST REPORT")
print("====================")

print(
    "Supported:",
    stats["supported"]
)

print(
    "Contradicted:",
    stats["contradicted"]
)

print(
    "Neutral:",
    stats["neutral"]
)

print(
    "Trust Score:",
    stats["trust_score"],
    "%"
)



TRUST REPORT
Supported: 0
Contradicted: 2
Neutral: 1
Trust Score: 16.67 %
